In [ ]:
!pip install xgboost


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
import shutil

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')


# Load the Dataset From Kaggle

In [ ]:

# Check if kaggle.json exists in Downloads
os.path.exists(r"C:\Users\Samuel\Downloads\kaggle.json")


# Create .kaggle directory
os.makedirs(r"C:\Users\Samuel\.kaggle", exist_ok=True)


# Move kaggle.json to .kaggle folder
shutil.move(
    r"C:\Users\Samuel\Downloads\kaggle.json",
    r"C:\Users\Samuel\.kaggle\kaggle.json"
)


# Verify kaggle.json is in the correct location
os.path.exists(r"C:\Users\Samuel\Downloads\kaggle.json")


# Download Home Credit dataset
!kaggle competitions download -c home-credit-default-risk



# Extract the dataset
with zipfile.ZipFile("home-credit-default-risk.zip", "r") as z:
    z.extractall("home_credit")


# List extracted files
os.listdir("home_credit")


# Read the Dataset

In [ ]:

# Must-use datasets
train = pd.read_csv("home_credit/application_train.csv")
test = pd.read_csv("home_credit/application_test.csv")


In [ ]:
print("Train shape:", train.shape)
print(train.head())
print("\nTarget distribution:")
print(train['TARGET'].value_counts(normalize=True))


In [ ]:
sns.countplot(x='TARGET', data=train)
plt.title("Loan Default Distribution")
plt.show()


# =============================================
2️. Data Preprocessing & Feature Engineering   
==============================================

#### 2.1 Handle INVALID VALUES (Home Credit specific)

In [ ]:
# DAYS_EMPLOYED has invalid placeholder value
train['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
test['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

# DAYS_BIRTH should be positive
train['DAYS_BIRTH'] = train['DAYS_BIRTH'].abs()
test['DAYS_BIRTH'] = test['DAYS_BIRTH'].abs()


### 2.2 Handle MISSING VALUES

In [ ]:
import pandas as pd
import numpy as np

# Identify numeric columns that exist in both train and test
num_cols = [col for col in train.select_dtypes(include=['int64','float64']).columns if col in test.columns]

# Identify categorical columns that exist in both train and test
cat_cols = [col for col in train.select_dtypes(include=['object']).columns if col in test.columns]

# Fill numeric missing values with median
for col in num_cols:
    train[col].fillna(train[col].median(), inplace=True)
    test[col].fillna(test[col].median(), inplace=True)

# Fill categorical missing values with mode
for col in cat_cols:
    train[col].fillna(train[col].mode()[0], inplace=True)
    test[col].fillna(test[col].mode()[0], inplace=True)

# Optional: check if any missing values remain
print("Remaining missing in train:", train.isnull().sum().sum())
print("Remaining missing in test:", test.isnull().sum().sum())


### 2.3 Handle EXTREME OUTLIERS (Winsorization / Clipping

In [ ]:
def clip_outliers(df, cols, lower=0.01, upper=0.99):
    for col in cols:
        low = df[col].quantile(lower)
        high = df[col].quantile(upper)
        df[col] = df[col].clip(low, high)
    return df

outlier_cols = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'DAYS_EMPLOYED',
    'DAYS_BIRTH'
]

train = clip_outliers(train, outlier_cols)
test = clip_outliers(test, outlier_cols)


### One-Hot Encoding (better for non-ordinal categories)

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns
cat_cols = train.select_dtypes(include='object').columns

#  Label encode binary categorical columns
for col in cat_cols:
    if train[col].nunique() == 2:
        le = LabelEncoder()  # new encoder for each column
        train[col] = le.fit_transform(train[col])
        test[col] = le.transform(test[col])

#  One-hot encode non-binary categorical columns
train = pd.get_dummies(train, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

# Align columns (validation set must match train)
test = test.reindex(columns=train.columns, fill_value=0)


### 2.5 Feature Engineering (Domain Knowledge)

In [ ]:
train['CREDIT_INCOME_PERCENT'] = train['AMT_CREDIT'] / train['AMT_INCOME_TOTAL']
train['ANNUITY_INCOME_PERCENT'] = train['AMT_ANNUITY'] / train['AMT_INCOME_TOTAL']
train['CREDIT_TERM'] = train['AMT_ANNUITY'] / train['AMT_CREDIT']
train['DAYS_EMPLOYED_PERCENT'] = train['DAYS_EMPLOYED'] / train['DAYS_BIRTH']




In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# -------------------------------
# 1️⃣ Handle placeholder values
# -------------------------------
train['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
test['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

train['DAYS_BIRTH'] = train['DAYS_BIRTH'].abs()
test['DAYS_BIRTH'] = test['DAYS_BIRTH'].abs()

# -------------------------------
# 2️⃣ Handle missing values
# -------------------------------
# Numeric columns
num_cols = train.select_dtypes(include=['int64','float64']).columns
num_cols = [c for c in num_cols if c in test.columns]  # exclude 'TARGET'

for col in num_cols:
    train[col].fillna(train[col].median(), inplace=True)
    test[col].fillna(test[col].median(), inplace=True)

# Categorical columns
cat_cols = train.select_dtypes(include='object').columns
cat_cols = [c for c in cat_cols if c in test.columns]

for col in cat_cols:
    train[col].fillna(train[col].mode()[0], inplace=True)
    test[col].fillna(test[col].mode()[0], inplace=True)

# -------------------------------
# 3️⃣ Handle outliers (optional)
# -------------------------------
def clip_outliers(df, cols, lower=0.01, upper=0.99):
    for col in cols:
        low = df[col].quantile(lower)
        high = df[col].quantile(upper)
        df[col] = df[col].clip(low, high)
    return df

outlier_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_EMPLOYED', 'DAYS_BIRTH']
train = clip_outliers(train, outlier_cols)
test = clip_outliers(test, outlier_cols)

# -------------------------------
# 4️⃣ Feature engineering (optional)
# -------------------------------
train['CREDIT_INCOME_PERCENT'] = train['AMT_CREDIT'] / train['AMT_INCOME_TOTAL']
train['ANNUITY_INCOME_PERCENT'] = train['AMT_ANNUITY'] / train['AMT_INCOME_TOTAL']
train['CREDIT_TERM'] = train['AMT_ANNUITY'] / train['AMT_CREDIT']
train['DAYS_EMPLOYED_PERCENT'] = train['DAYS_EMPLOYED'] / train['DAYS_BIRTH']

test['CREDIT_INCOME_PERCENT'] = test['AMT_CREDIT'] / test['AMT_INCOME_TOTAL']
test['ANNUITY_INCOME_PERCENT'] = test['AMT_ANNUITY'] / test['AMT_INCOME_TOTAL']
test['CREDIT_TERM'] = test['AMT_ANNUITY'] / test['AMT_CREDIT']
test['DAYS_EMPLOYED_PERCENT'] = test['DAYS_EMPLOYED'] / test['DAYS_BIRTH']

# -------------------------------
# 5️⃣ Categorical encoding
# -------------------------------
X_train = train.drop(columns=['TARGET'])
y_train = train['TARGET']
X_valid = test.copy()  # assuming test is used as validation here

cat_cols = X_train.select_dtypes(include='object').columns

# Label encode binary columns
for col in cat_cols:
    if X_train[col].nunique() == 2:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col])
        X_valid[col] = le.transform(X_valid[col])

# One-hot encode multi-class columns
X_train = pd.get_dummies(X_train, drop_first=True)
X_valid = pd.get_dummies(X_valid, drop_first=True)

# Align validation columns
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)

# =====================================================
3️ Model Implementation
=======================================================

### important liberaries for model implementation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


In [ ]:
y = train['TARGET']
X = train.drop(columns=['TARGET'])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


### Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
     min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
rf_preds = rf.predict(X_valid)


### XGBoost Model

In [ ]:
neg = (train == 0).sum()
pos = (train == 1).sum()

scale_pos_weight = neg / pos


xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    tree_method="hist"
)


xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_valid)


# =====================================================
4️. Model Evaluation & Results
=======================================================

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
def evaluate_model(name, y_true, y_pred):
    print(f"{name} Results")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1-score :", f1_score(y_true, y_pred))
    print("-" * 40)


In [ ]:
evaluate_model("Random Forest", y_valid, rf_preds)
evaluate_model("XGBoost", y_valid, xgb_preds)


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_valid, rf_preds),
        accuracy_score(y_valid, xgb_preds)
    ],
    "Precision": [
        precision_score(y_valid, rf_preds),
        precision_score(y_valid, xgb_preds)
    ],
    "Recall": [
        recall_score(y_valid, rf_preds),
        recall_score(y_valid, xgb_preds)
    ],
    "F1 Score": [
        f1_score(y_valid, rf_preds),
        f1_score(y_valid, xgb_preds)
    ]
})

comparison


# =====================================================
5️. Explainability / Fairness (SHAP)
=======================================================

In [ ]:
import shap
shap.initjs()


In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_valid)


In [ ]:
# Global explanation
shap.summary_plot(shap_values, X_valid)


In [ ]:
# Local explanation for one instance
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_valid.iloc[0],
    matplotlib=True
)
